# 05 Extension Frameworks

This notebook is a holding place for topics that belong to Chapter 2 but are not
the main priority of the first build. The code cells are intentionally small and
serve as runnable sketches for later expansion.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


## Cubic Hermite interpolation

Hermite interpolation uses values and derivatives. On one interval, the cubic
Hermite form can be written with four basis functions:

$$
p(t)=h_{00}(t)y_0+h_{10}(t)m_0+h_{01}(t)y_1+h_{11}(t)m_1,
\qquad 0\le t\le 1.
$$

A later pass should connect these basis functions to PCHIP slope selection.


In [ ]:
def hermite_basis(t):
    t = np.asarray(t, dtype=float)
    h00 = 2 * t**3 - 3 * t**2 + 1
    h10 = t**3 - 2 * t**2 + t
    h01 = -2 * t**3 + 3 * t**2
    h11 = t**3 - t**2
    return h00, h10, h01, h11

t = np.linspace(0.0, 1.0, 300)
labels = ["h00", "h10", "h01", "h11"]

plt.figure(figsize=(8, 4.5))
for values, label in zip(hermite_basis(t), labels):
    plt.plot(t, values, label=label)
plt.xlabel("t")
plt.ylabel("basis value")
plt.title("Cubic Hermite basis functions")
plt.legend()
plt.tight_layout()
plt.show()


## PCHIP framework

PCHIP is a piecewise cubic Hermite method with carefully chosen slopes. Its goal
is to preserve monotonicity and avoid overshoot. This chapter should later add:

* slope estimates from neighboring secant slopes;
* cases where slopes are set to zero to preserve monotonicity;
* comparison with natural cubic splines on monotone data.


## Cubic uniform B-spline framework

B-splines use local basis functions. A cubic uniform B-spline basis has compact
support over four unit intervals. Local support is the main idea to preserve for
later expansion.


In [ ]:
def truncated_power_3(x):
    return np.maximum(x, 0.0) ** 3


def cubic_uniform_b_spline_basis(x):
    return (
        truncated_power_3(x)
        - 4 * truncated_power_3(x - 1)
        + 6 * truncated_power_3(x - 2)
        - 4 * truncated_power_3(x - 3)
        + truncated_power_3(x - 4)
    ) / 6.0

x_basis = np.linspace(-1.0, 6.0, 700)

plt.figure(figsize=(8, 4.5))
for shift in range(4):
    plt.plot(x_basis, cubic_uniform_b_spline_basis(x_basis - shift), label=f"B(x-{shift})")
plt.xlabel("x")
plt.ylabel("basis value")
plt.title("Translated cubic uniform B-spline basis functions")
plt.legend()
plt.tight_layout()
plt.show()


## Two-dimensional interpolation framework

The first two-dimensional examples should stay simple:

* bilinear interpolation on a rectangle;
* linear Lagrange interpolation on a triangle.

These are enough to connect the chapter to grids, images, and finite elements.


In [ ]:
def bilinear_unit_square(x, y, z00, z10, z01, z11):
    return (
        (1 - x) * (1 - y) * z00
        + x * (1 - y) * z10
        + (1 - x) * y * z01
        + x * y * z11
    )

u = np.linspace(0.0, 1.0, 80)
v = np.linspace(0.0, 1.0, 80)
U, V = np.meshgrid(u, v)
Z = bilinear_unit_square(U, V, z00=0.0, z10=1.0, z01=0.5, z11=0.2)

plt.figure(figsize=(6, 5))
contours = plt.contourf(U, V, Z, levels=20)
plt.colorbar(contours, label="interpolated value")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Bilinear interpolation on a unit square")
plt.tight_layout()
plt.show()


In [ ]:
def barycentric_coordinates(point, triangle):
    p = np.asarray(point, dtype=float)
    tri = np.asarray(triangle, dtype=float)
    matrix = np.array([
        [tri[0, 0], tri[1, 0], tri[2, 0]],
        [tri[0, 1], tri[1, 1], tri[2, 1]],
        [1.0, 1.0, 1.0],
    ])
    rhs = np.array([p[0], p[1], 1.0])
    return np.linalg.solve(matrix, rhs)

triangle = np.array([[0.0, 0.0], [1.0, 0.0], [0.2, 0.9]])
values = np.array([1.0, 2.0, 0.5])
point = np.array([0.35, 0.25])
lambdas = barycentric_coordinates(point, triangle)
interpolated_value = lambdas @ values

print("barycentric coordinates:", lambdas)
print("interpolated value:", interpolated_value)


## Extension summary

Hermite interpolation, PCHIP, B-splines, and two-dimensional interpolation are
part of the Chapter 2 roadmap. They are intentionally introduced here as
frameworks so later updates can deepen them without disrupting the completed
main path.
